# Data Cleaning Assessment – Solution Copy
## Instructor Answer Key


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_student_data_80k.csv')

# 1. Rename columns
df.rename(columns={
    'Student ID': 'student_id',
    'Full Name': 'student_name',
    'Contact Email': 'email',
    'Age': 'age',
    'Gender': 'gender',
    'City': 'city',
    'Department': 'department',
    'Degree Course': 'course',
    'Marks Percentage': 'marks',
    'Annual Package': 'annual_salary',
    'Mobile Number': 'phone',
    'Enrollment Date': 'admission_date'
}, inplace=True)

# 2. Replace missing tokens
missing_tokens = ['', 'NA', 'N/A', 'null', 'Not Available', 'unknown']
df = df.replace(missing_tokens, np.nan)

# 3. Categorical & Text cleaning
cat_cols = ['gender', 'city', 'department', 'course']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown').astype(str).str.strip().str.lower()

df['city'] = df['city'].replace({'bangalore': 'bengaluru', 'gurgaon': 'gurugram'})
df['department'] = df['department'].replace({
    'computer science': 'cse', 'electronics': 'ece',
    'information technology': 'it', 'mechanical': 'me'
})
df['course'] = df['course'].str.replace('.','',regex=False).str.replace(' ','',regex=False)

# 4. Drop Duplicates
df.drop_duplicates(inplace=True)

# 5. Numeric validation & imputation
for col in ['age', 'marks', 'annual_salary']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[~df['age'].between(17, 35), 'age'] = np.nan
df.loc[~df['marks'].between(0, 100), 'marks'] = np.nan
df.loc[(df['annual_salary'] <= 0) | (df['annual_salary'] > 10000000), 'annual_salary'] = np.nan

for col in ['age', 'marks', 'annual_salary']:
    df[col] = df[col].fillna(df[col].median())

# 6. Emails & Phones
df['email'] = df['email'].astype(str).str.strip().str.lower()
valid_email = df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+$', na=False)
df.loc[~valid_email, 'email'] = np.nan

df['phone'] = df['phone'].astype(str).str.replace(r'\D','',regex=True)
valid_phone = df['phone'].str.match(r'^[6-9]\d{9}$', na=False)
df.loc[~valid_phone, 'phone'] = np.nan

# 7. Parse Date and Export
df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')
df.to_csv('clean_student_data_80k.csv', index=False)
print('SUCCESS: Cleaning complete and saved to clean_student_data_80k.csv')